# CoPaw-Flash-9B LoRA SFT — Runbook (M3 24GB)

Self-contained notebook for the fine-tuning workflow in this repo. Run cells top-to-bottom instead of re-deriving setup with an LLM.

**Model:** [`andjiang/CoPaw-Flash-9B-oQ4`](https://huggingface.co/andjiang/CoPaw-Flash-9B-oQ4) (Qwen3.5 9B, 4-bit, 24/32 GatedDeltaNet linear-attention layers)

**Sibling data repo:** `../Finetuning-data-workflows/llm-usage-extract` (judge results + replay pool)

**Key scripts:**
- `train_sft.py` — build ChatML split, write LoRA config, orchestrate training
- `train_runner.py` — **required** memory-safe wrapper around `mlx_lm.lora` (do not call `mlx_lm lora` directly on this model)
- `chatml.py` — manual ChatML renderer (handles episodes with no leading user turn)
- `test_chunked_patch.py` — numerical equivalence tests for the manual backprop engine

## 0. Configuration

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
PIPELINE_ROOT = REPO.parent / "Finetuning-data-workflows" / "llm-usage-extract"

# Use the data-pipeline venv (has mlx-lm + pipeline modules)
VENV_PY = PIPELINE_ROOT / ".venv" / "bin" / "python3"
if not VENV_PY.exists():
    VENV_PY = Path(sys.executable)

MODEL = "andjiang/CoPaw-Flash-9B-oQ4"
MAX_SEQ_LENGTH = 8192  # current working value; see token-length section below
NUM_LAYERS = 16          # last 16 of 32 blocks (mlx_lm default); -1 = all layers
EPOCHS = 2.0
RANK = 16
ALPHA = 16.0
LR = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
SEED = 0

# oMLX / unified-memory budget on this M3 (do not exceed ceil — prior run froze the machine)
OMLX_SOFT_GB = 17.8
OMLX_HARD_GB = 19.9
OMLX_CEIL_GB = 21.0

CANARY_ITERS = 3  # uses the N longest training examples (worst-case memory probe)

print(f"Repo:           {REPO}")
print(f"Pipeline root:  {PIPELINE_ROOT}")
print(f"Python:         {VENV_PY}")
print(f"max_seq_length: {MAX_SEQ_LENGTH}")

## 1. Prerequisites check

In [ ]:
required = {
    "judge_results.jsonl": PIPELINE_ROOT / "output" / "judge_results.jsonl",
    "replay_filtered.jsonl": PIPELINE_ROOT / "output" / "replay_filtered.jsonl",
    "train_sft.py": REPO / "train_sft.py",
    "train_runner.py": REPO / "train_runner.py",
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing: {missing}")

subprocess.run([str(VENV_PY), "-c", "import mlx_lm; print('mlx_lm OK')"], check=True)
print("All prerequisites present.")

## 2. Data inventory (Schritt 0 from Plan v4)

Token length drives memory more than example count for agent trajectories. The default `max_seq_length=32768` keeps 95.8% of domain episodes but is too long for 24GB with this architecture.

In [ ]:
sys.path.insert(0, str(PIPELINE_ROOT))
import build_sft
from mlx_lm.utils import load_tokenizer
from chatml import render_chatml
from train_sft import (
    build_domain_rows,
    load_nudge_flags,
    token_length,
)

domain_input = PIPELINE_ROOT / "output" / "judge_results.jsonl"
nudge_flags = load_nudge_flags(PIPELINE_ROOT / "output" / "nudge_flags.jsonl")
domain_rows = build_domain_rows(build_sft, domain_input, coherence_floor=3, nudge_flags=nudge_flags)
tokenizer = load_tokenizer(MODEL)
lengths = sorted(token_length(tokenizer, r["text"]) for r in domain_rows)

def pct(n: int) -> float:
    return 100.0 * n / len(lengths)

for limit in (8192, 16384, 24576, 32768):
    kept = sum(1 for L in lengths if L <= limit)
    print(f"max_seq_length={limit:5d}: keep {kept:3d}/{len(lengths)} ({pct(kept):5.1f}%)")

mid = lengths[len(lengths) // 2]
p90 = lengths[int(0.9 * len(lengths))]
print(f"\nDomain episodes accepted: {len(domain_rows)}")
print(f"Token lengths — min={lengths[0]}, median={mid}, p90={p90}, max={lengths[-1]}")

dpo_path = PIPELINE_ROOT / "output" / "dpo_pairs.jsonl"
print(f"\nDPO pairs exist: {dpo_path.exists()} (staged_dpo mode needs these)")

## 3. Why `train_runner.py` exists (memory learnings)

**Do not run `python -m mlx_lm lora` directly** on CoPaw-Flash-9B. The first backward pass peaks at ~21GB+ regardless of `max_seq_length`.

| Approach | Result at T=8192 |
|---|---|
| Stock `mlx_lm.lora` GatedDeltaNet backward (ops fallback) | ~21GB+, OOM / freeze |
| `mx.checkpoint` chunking | **No help** — MLX runs recomputes concurrently |
| mlx_lm `grad_checkpoint` | Same problem |
| Custom VJP with `mx.depends` sequencing | Peak grows linearly with T (66GB at T=8192 for one layer) |
| **`train_runner.py` manual BPTT + `mx.eval()` boundaries** | ~16.4GB MLX peak, ~18–19GB `phys_footprint_peak` |

The fix splits backprop into Python-level segments with explicit `mx.eval()` between them:
- **Phase F:** layer-by-layer forward (kernel scans, no grads)
- **Phase C:** chunked cross-entropy (1024 tokens/chunk; avoids 248k-vocab logits)
- **Phase B:** reverse layer backward; GatedDeltaNet uses chunkwise-parallel scan (matmuls, not 128 sequential steps)

Other guardrails:
- Wired limit clamped to **14GB** (stock `train()` can wire ~22GB after sysctl bump → system freeze)
- Output head must stay **frozen** (LoRA only)
- Monitor memory with **`footprint <pid>`** (`phys_footprint_peak`), not `ps rss` (undercounts GPU allocations by ~3×)

Run numerical tests anytime:
```bash
python test_chunked_patch.py
```

## 4. Build split + config (dry run)

In [ ]:
cmd = [
    str(VENV_PY), str(REPO / "train_sft.py"),
    "--pipeline-root", str(PIPELINE_ROOT),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--num-layers", str(NUM_LAYERS),
    "--epochs", str(EPOCHS),
    "--rank", str(RANK),
    "--alpha", str(ALPHA),
    "--learning-rate", str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accumulation-steps", str(GRAD_ACCUM),
    "--seed", str(SEED),
    "--dry-run",
]
subprocess.run(cmd, cwd=REPO, check=True)

## 5. Canary memory probe (always run before a full training)

Uses the **longest** training examples (not a random sample). Kills the process if `phys_footprint_peak` hits the 21GB ceiling.

**Observed at max_seq_length=8192, num_layers=16 (Jul 2026):**
- Peak RSS: **18.0 GB** (v5 canary, 3 iters)
- MLX peak mem: **16.4 GB**
- ~**90 s/iter** on worst-case ~8k-token examples → full 978-iter run ≈ **24+ hours** (before eval overhead)

Decision thresholds (printed by `train_sft.py`):
- ≥ hard (19.9 GB): do not train at this length
- ≥ soft (17.8 GB): risky — consider lowering `max_seq_length`
- < soft: proceed if you accept the speed cost

In [ ]:
canary_cmd = [
    str(VENV_PY), str(REPO / "train_sft.py"),
    "--pipeline-root", str(PIPELINE_ROOT),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--num-layers", str(NUM_LAYERS),
    "--epochs", str(EPOCHS),
    "--rank", str(RANK),
    "--alpha", str(ALPHA),
    "--learning-rate", str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accumulation-steps", str(GRAD_ACCUM),
    "--seed", str(SEED),
    "--canary-only",
    "--canary-iters", str(CANARY_ITERS),
]
print("Running canary (may take ~15–30 min)...")
subprocess.run(canary_cmd, cwd=REPO, check=True)

## 6. Full LoRA training

Only run after the canary passes your memory budget. Uses `train_runner.py` under the hood.

Crash recovery: re-run with `--resume` (restores adapter **weights only**, not Adam state).

In [ ]:
RUN_FULL = False  # flip to True after reviewing canary output

if not RUN_FULL:
    print("Set RUN_FULL = True to start training.")
else:
    train_cmd = [
        str(VENV_PY), str(REPO / "train_sft.py"),
        "--pipeline-root", str(PIPELINE_ROOT),
        "--max-seq-length", str(MAX_SEQ_LENGTH),
        "--num-layers", str(NUM_LAYERS),
        "--epochs", str(EPOCHS),
        "--rank", str(RANK),
        "--alpha", str(ALPHA),
        "--learning-rate", str(LR),
        "--batch-size", str(BATCH_SIZE),
        "--grad-accumulation-steps", str(GRAD_ACCUM),
        "--seed", str(SEED),
        # "--resume",  # uncomment after a crash
    ]
    subprocess.run(train_cmd, cwd=REPO, check=True)

## 7. Hyperparameter reference (Plan v4 defaults)

| Parameter | Value | Notes |
|---|---|---|
| `fine_tune_type` | lora | QLoRA-style on 4-bit base |
| `rank` / `scale` | 16 / 1.0 | α=16, 1:1 scaling |
| `target_modules` | all-linear | via `num_layers=-1`; we use 16 for stability |
| `learning_rate` | 1e-4 | |
| `epochs` | 2 | ~978 iters at 489 train rows, batch=1 |
| `grad_checkpoint` | true | trades memory for Metal resource pressure |
| `grad_accumulation_steps` | 8 | effective batch ≈ 8 |
| `save_every` | 25 | checkpoints `{iter:07d}_adapters.safetensors` |
| `val_batches` | 8 | full eval at 8k tokens would take >1h |
| Eval split | 10% random domain | `completion_status` not in judge schema; nudge stratification unavailable |

**Outputs (gitignored):**
- `output/mlx_data/{train,valid}.jsonl` — ChatML training rows
- `output/lora_config.yaml` — mlx_lm config
- `output/adapters/adapters.safetensors` — final adapter (~82 MB)
- `output/adapters/*_adapters.safetensors` — numbered checkpoints

## 8. Speed expectations & open problems

- Training is **correct but slow** (~0.003–0.004 it/s on 8k-token examples). A full 2-epoch run at 8192 is **~1 day minimum**, not 4 days, but still impractical for iteration.
- Chunkwise-parallel GatedDeltaNet backward (matmul formulation) was the main speed win over per-timestep Python loops; further gains likely need a different framework or fused Metal kernel with VJP support.
- `num_layers=16` + `max_seq_length=8192` is the current stable config; pushing to 32768 requires either more RAM or architectural changes.
- DPO stage (Plan v4 Option 3) is **not implemented** here; no `dpo_pairs.jsonl` exists yet.

**CLI equivalents** (same as this notebook):
```bash
# Dry run
python train_sft.py --dry-run --max-seq-length 8192 --num-layers 16

# Canary
python train_sft.py --canary-only --canary-iters 3 --max-seq-length 8192 --num-layers 16

# Full run
python train_sft.py --max-seq-length 8192 --num-layers 16

# Resume after crash
python train_sft.py --max-seq-length 8192 --num-layers 16 --resume
```